# Live webcam

The eye model runs on a webcam stream (or a video file) as fast as it can. See `eyemodel3d/live.py`.

- **`MODE = "face"`:** OpenCV's YuNet face detector finds both eyes. Each eye is cropped like the
  training inputs (iris about 20% of the width, eyes level), and both are predicted in one batch.
- **`MODE = "center"`:** a fixed box in the middle of the frame. Hold one eye close to the
  camera to try close-ups.

On the video: outlines are white = sclera, cyan = iris, yellow = pupil, magenta = caruncle,
with the predicted laterality and P(OD) above each eye. The panel on the right shows each
network input and its prediction.

In a plain (non-mirrored) webcam image, your right eye (OD) is on the image *left*. With
`MIRROR = True`, the preview acts like a mirror and the expected sides swap.

The camera is opened by the kernel, so it must be plugged into the machine running the kernel.
`SOURCE` can also be a video file path (it then loops).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import glob

import ipywidgets as wd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from eyemodel3d import live as L

# ---- edit these
CKPT = ROOT / "runs/vit_s_pupil/best.pt"
SOURCE = 0  # camera index (/dev/videoN), or a video file path
MODE = "face"  # "face" or "center"
SMOOTH = 0.5  # weight of the past in the temporal smoothing (0 = none)
MIRROR = False  # flip the frames horizontally (selfie view)
WIDTH = 640  # frames are resized to this width

cams = sorted(glob.glob("/dev/video*"))
print("cameras:", cams or "none found (plug one in, or set SOURCE to a video file)")

In [ ]:
tracker = L.EyeTracker(CKPT, mode=MODE, smooth=SMOOTH)
m = tracker.meta
print(f"{CKPT.relative_to(ROOT)}: epoch {m['epoch']}, val Dice {m['val_dice']:.3f}, "
      f"{m['backbone']}, pupil {'predicted' if tracker.predicts_pupil else 'not predicted'}, "
      f"device {tracker.dev}")

## Run
**Start** / **Stop** control the stream. Mode and smoothing can be changed while it runs.

In [ ]:
view = wd.Image(format="jpeg", layout=wd.Layout(width="960px"))
status = wd.HTML()
b_start = wd.Button(description="Start", button_style="success", icon="play")
b_stop = wd.Button(description="Stop", button_style="danger", icon="stop")
w_mode = wd.ToggleButtons(options=["face", "center"], value=MODE, description="mode")
w_smooth = wd.FloatSlider(value=SMOOTH, min=0, max=0.95, step=0.05, description="smoothing")
w_center = wd.FloatSlider(value=tracker.center_frac, min=0.2, max=1.0, step=0.05,
                          description="box size", tooltip="center mode: box height / frame height")
w_mirror = wd.Checkbox(value=MIRROR, description="mirror")


def show(jpeg, results):
    view.value = jpeg
    eyes = ", ".join(f"{r['lat']} ({r['p_od']:.2f})" for r in results) or "no eye"
    status.value = f"{loop.fps:.1f} fps &nbsp; {eyes}"


def ended(err):
    status.value = f"<b style='color:red'>stopped: {err}</b>" if err else "stopped"


loop = L.LiveLoop(tracker, SOURCE, show=show, width=WIDTH, mirror=MIRROR, on_end=ended)


def start(_):
    loop.mirror = w_mirror.value
    loop.start()


def set_mode(ch):
    tracker.mode = ch["new"]
    tracker._z = tracker._eyes = None


b_start.on_click(start)
b_stop.on_click(lambda _: loop.stop())
w_mode.observe(set_mode, "value")
w_smooth.observe(lambda ch: setattr(tracker, "smooth", ch["new"]), "value")
w_center.observe(lambda ch: setattr(tracker, "center_frac", ch["new"]), "value")
w_mirror.observe(lambda ch: setattr(loop, "mirror", ch["new"]), "value")

display(wd.VBox([wd.HBox([b_start, b_stop, w_mode, w_mirror]), wd.HBox([w_smooth, w_center]),
                 status, view]))

## Snapshot
Details of the last processed frame: each eye's network input, prediction and parameters.
Run it while the stream is running, or after **Stop**.

In [ ]:
if loop.last is None:
    print("no frame processed yet")
else:
    frame, results = loop.last
    fig, axs = plt.subplots(max(1, len(results)), 2, figsize=(8, 3.2 * max(1, len(results))),
                            squeeze=False)
    for r, (a0, a1) in zip(results, axs):
        a0.imshow(r["crop"])
        a0.set_title(f"network input ({r['lat']}, P(OD) {r['p_od']:.2f})", fontsize=9)
        lbl = r["lbl"]
        rgb = L.T.colorize(np.where(lbl == L.PUPIL, L.T.IRIS, lbl))
        rgb[lbl == L.PUPIL] = L.em.COLORS[L.em.CLASSES.index("pupil")].numpy()
        a1.imshow(rgb)
        a1.set_title("prediction", fontsize=9)
        for a in (a0, a1):
            a.set_axis_off()
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame({f"eye {k} ({r['lat']})": r["vals"] for k, r in enumerate(results)})
            .astype(float).round(3))

## Stop and release the camera

In [ ]:
loop.stop()